# 03 · Gradio interfaces

Gradio turns Python callables into web UIs: it generates a Svelte frontend, serves it from a FastAPI
backend, and exposes each callback as an HTTP endpoint (plus an auto-generated API client).

| API | Use for |
|---|---|
| `gr.Interface` | one function, inputs → outputs |
| `gr.ChatInterface` | chat UIs with history management built in |
| `gr.Blocks` | custom layouts and event wiring between components |

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, for `toolkit`

In [ ]:
import gradio as gr

from toolkit import available_providers, stream
from toolkit.chat_ui import accumulate

## Interface: any function becomes a UI

In [ ]:
def word_stats(text: str) -> str:
    words = text.split()
    return f"{len(words)} words, {len(set(w.lower() for w in words))} unique, {len(text)} chars"


gr.Interface(fn=word_stats, inputs="textbox", outputs="textbox", flagging_mode="never").launch()

## Streaming + model selection

A generator callback streams. Gradio replaces the output on every `yield`, so the callback yields
the running text rather than individual deltas (`toolkit.chat_ui.accumulate`).

In [ ]:
SYSTEM = "You are a concise technical writer. Respond in markdown, not wrapped in a code fence."


def explain(prompt, provider):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}]
    yield from accumulate(stream(messages, provider=provider))


providers = available_providers()
gr.Interface(
    fn=explain,
    inputs=[
        gr.Textbox(label="Prompt", lines=4),
        gr.Dropdown(providers, value=providers[0], label="Provider"),
    ],
    outputs=gr.Markdown(),
    examples=[
        ["Explain the transformer architecture to a product manager", providers[0]],
        ["Explain the transformer architecture to a backend engineer", providers[0]],
    ],
    flagging_mode="never",
).launch()

## Blocks: custom layout and chained events

Side-by-side comparison of two providers on the same prompt. `.then()` chains events;
the two generators run one after the other.

In [ ]:
with gr.Blocks(title="Compare") as compare:
    prompt = gr.Textbox(label="Prompt")
    with gr.Row():
        left_provider = gr.Dropdown(providers, value=providers[0], label="Left")
        right_provider = gr.Dropdown(providers, value=providers[-1], label="Right")
    with gr.Row():
        left, right = gr.Markdown(), gr.Markdown()

    prompt.submit(explain, [prompt, left_provider], left).then(
        explain, [prompt, right_provider], right
    )

compare.launch()

Launch options worth knowing: `share=True` (temporary public tunnel), `auth=(user, password)`
(basic auth; credentials belong in env vars), `inbrowser=True`. Hosting long-term: Hugging Face Spaces.